# Index restoration utility

Restore the pre-configured `supplier-evidence` and `sourcing-documents` indexes used throughout ILL344. The notebooks assume these indexes already exist in the search service configured by `AZURE_SEARCH_SERVICE_ENDPOINT`.

In [ ]:
import json
import os
import traceback
from typing import Any, cast

from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.aio import SearchClient
from azure.search.documents.indexes.aio import SearchIndexClient
from azure.search.documents.indexes.models import SearchIndex

load_dotenv(override=True)
SEARCH_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
SEARCH_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]

credential = AzureKeyCredential(SEARCH_KEY)
print(f"Search endpoint : {SEARCH_ENDPOINT}")
print("Search auth     : api-key")
print(f"OpenAI endpoint : {AZURE_OPENAI_ENDPOINT}")

In [ ]:
async def restore_index(endpoint: str, credential, index_name: str, index_file: str, records_file: str, azure_openai_endpoint: str):
    default_path = r"../../data/index-data"
    try:
        print(f"[{index_name}] Starting index restoration...")
        async with SearchIndexClient(endpoint=endpoint, credential=credential) as client:
            index_file_path = os.path.join(default_path, index_file)
            print(f"[{index_name}] Reading index definition from: {index_file_path}")
            with open(index_file_path, "r", encoding="utf-8") as in_file:
                index = cast(Any, SearchIndex)._deserialize(json.load(in_file), [])
                index.name = index_name
                index.vector_search.vectorizers[0].parameters.resource_url = azure_openai_endpoint
                index.vector_search.vectorizers[0].parameters.api_key = AZURE_OPENAI_KEY
                await client.create_or_update_index(index)
        async with SearchClient(endpoint=endpoint, index_name=index_name, credential=credential) as client:
            records_file_path = os.path.join(default_path, records_file)
            print(f"[{index_name}] Reading documents from: {records_file_path}")

            existing_results = await client.search(search_text="*", select=["chunk_id"])
            existing_records = [{"chunk_id": result["chunk_id"]} async for result in existing_results]
            for offset in range(0, len(existing_records), 1000):
                results = await client.delete_documents(documents=existing_records[offset : offset + 1000])
                if any(not result.succeeded for result in results):
                    raise RuntimeError(f"Failed to remove stale documents from {index_name}")
            if existing_records:
                print(f"[{index_name}] Removed {len(existing_records)} stale documents")

            records, expected_count, total_uploaded, batch_count = [], 0, 0, 0

            async def upload_batch(batch):
                nonlocal batch_count, total_uploaded
                batch_count += 1
                print(f"[{index_name}] Uploading batch #{batch_count} ({len(batch)} documents)...")
                results = await client.upload_documents(documents=batch)
                failures = [result for result in results if not result.succeeded]
                if failures:
                    details = ", ".join(f"{result.key}: {result.error_message}" for result in failures)
                    raise RuntimeError(f"Document upload failed: {details}")
                total_uploaded += len(results)

            with open(records_file_path, "r", encoding="utf-8") as in_file:
                for line_num, line in enumerate(in_file, 1):
                    try:
                        records.append(json.loads(line))
                        expected_count += 1
                    except json.JSONDecodeError as error:
                        raise ValueError(
                            f"Invalid JSON in {records_file_path} on line {line_num}: {error}"
                        ) from error
                    if len(records) >= 100:
                        await upload_batch(records)
                        records = []
            if records:
                await upload_batch(records)
            if total_uploaded != expected_count:
                raise RuntimeError(f"Uploaded {total_uploaded} documents, expected {expected_count}")
            restored_count = await client.get_document_count()
            if restored_count != expected_count:
                raise RuntimeError(
                    f"Restored index contains {restored_count} documents, expected {expected_count}"
                )
        print(f"[{index_name}] SUCCESS - Expected and uploaded documents: {total_uploaded}")
        return True
    except Exception as error:
        print(f"[{index_name}] ERROR - {type(error).__name__}: {error}")
        traceback.print_exc()
        return False

In [ ]:
print("\n--- Processing supplier-evidence index ---")
await restore_index(
    SEARCH_ENDPOINT,
    credential,
    "supplier-evidence",
    "supplier-evidence-index.json",
    "supplier-evidence-exported.jsonl",
    AZURE_OPENAI_ENDPOINT,
)

In [ ]:
print("\n--- Processing sourcing-documents index ---")
await restore_index(
    SEARCH_ENDPOINT,
    credential,
    "sourcing-documents",
    "sourcing-documents-index.json",
    "sourcing-documents-exported.jsonl",
    AZURE_OPENAI_ENDPOINT,
)

## Next steps

Once both indexes are restored successfully, start with [Part 1: Multi-source KB with restored search indexes](part1-standard-foundry-iq-kb.ipynb).